In [ ]:
    ############    #############   FastAPI Architecture   #############   ##############   

 =>  FastAPI is built on three ideas that matter for production use: Starlette underneath
       for the actual ASGI request/response handling, Pydantic for request/response
       validation, and Python type hints doubling as the source of truth for both.

 =>  A route's type hints aren't just documentation -- FastAPI reads them at runtime to
       validate incoming data, serialize outgoing data, AND generate the OpenAPI schema
       (see the API contract standards notebook) -- one signature, three jobs.

 =>  ASGI (vs the older WSGI) is what makes native async support possible -- a route can be
       'async def' and actually benefit from non-blocking I/O (see Phase 0.1's AsyncIO
       notebook).


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI(title="FDE Academy Demo API")

class Ping(BaseModel):
    message: str

@app.get("/health", response_model=Ping)
async def health() -> Ping:
    return Ping(message="ok")

client = TestClient(app)
response = client.get("/health")
print(response.status_code, response.json())


In [ ]:
 =>  'response_model=Ping' does two things at once: it validates that whatever the
       function returns actually matches the Ping shape, and it tells FastAPI what to put
       in the OpenAPI schema for this route's response.

 =>  TestClient (built on httpx) lets you test routes directly, in-process, with no real
       network call or running server -- this is how you'll test every FastAPI route in
       this topic.


In [ ]:
    ############    #############   Structuring a Project for Maintainability and Clear Ownership   #############   ##############   

 =>  A single main.py with every route, model, and DB call inline works for a demo and
       becomes unreviewable within weeks. Split by responsibility, not by convenience:

           app/
             main.py          <- creates the FastAPI() app, wires routers, lifespan
             routers/         <- one module per resource area (users.py, orders.py, ...)
             services/        <- business logic, framework-agnostic
             repositories/    <- data access only
             models/          <- Pydantic request/response schemas
             config.py        <- Settings (see Phase 0.1's Configuration notebook)

 =>  'Clear ownership' means: given any bug report, you can point at exactly one file/module
       responsible for that behavior -- if fixing a bug requires touching 4 unrelated files,
       that's a sign responsibilities are tangled, not separated.

 =>  The next notebook (Routers, services, repositories) is this same idea taken one level
       deeper -- the folder layout here is what that layering looks like on disk.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Visit /docs on a real running FastAPI app (uvicorn main:app --reload) and see the
           interactive OpenAPI UI generated purely from your type hints.

 =>  [ ] Add a second route that takes a Pydantic request body, and confirm FastAPI returns
           a 422 automatically when you send invalid JSON.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Mixing 'def' and 'async def' routes without understanding the difference -- a 'def'
       route runs in a threadpool (fine for blocking code); an 'async def' route runs
       directly on the event loop (must never block it, see Phase 0.1's AsyncIO notebook).

 =>  Skipping response_model 'for speed' -- without it, a bug that leaks an internal field
       (e.g. a password hash) straight into the JSON response goes uncaught.
